# Variables temporales

Este notebook decide y construye las variables que describen el tiempo: qué pasó con el frío en los últimos días y en qué momento del año cae el día que se quiere predecir. Cada número que aparece en las tablas sale de una celda que se puede volver a ejecutar, y toda decisión se mide solo con `X_train`, nunca con validación ni prueba. La tabla de partida es la ya limpia, con las reglas aprendidas en entrenamiento, y las variables nuevas se calculan sobre la tabla completa en orden cronológico, usando únicamente información de hoy o anterior. Las justificaciones resumidas están en el ADR 0012.

## 1. Preparación: tabla limpia y división

Se carga la tabla completa (la primera ejecución tarda varios minutos y las siguientes, segundos, porque queda guardada en `data/interim/`), se divide en entrenamiento, validación y prueba como en el notebook de preprocesamiento, se aprenden las reglas de limpieza solo con entrenamiento y se aplican a toda la tabla. El control final debe coincidir con el de ese notebook: 1.212 días y 276 columnas.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

sys.path.append("../src")
from auxiliar_functions import tabla_simple, mostrar_lado_a_lado_simple, medir_efecto_por_grupo
from preprocessing_pipeline import cargar_tabla_completa, dividir_por_fecha
from limpieza import aprender_reglas_de_limpieza, limpiar
from variables_temporales import (crear_variables_temporales, es_fin_de_anio, NOMBRES_NUEVAS, COLUMNA_FRIO,
                                  VENTANA_DIAS, MIN_DIAS_CON_DATO, INICIO_FIN_DE_ANIO, FIN_FIN_DE_ANIO)

tabla_completa = cargar_tabla_completa(Path("../data"))
X_train, X_val, X_test = dividir_por_fecha(tabla_completa)
reglas = aprender_reglas_de_limpieza(X_train)            # reglas medidas solo con entrenamiento
tabla_limpia = limpiar(tabla_completa, reglas).copy()    # .copy() defragmenta la tabla que deja limpiar
train_limpio, val_limpio, test_limpio = dividir_por_fecha(tabla_limpia)

control = pd.DataFrame({"Valor": {
    "Días (filas) de la tabla limpia": len(tabla_limpia),
    "Columnas de la tabla limpia": tabla_limpia.shape[1],
    "Días de entrenamiento": len(train_limpio),
    "Días de validación": len(val_limpio),
    "Días de prueba": len(test_limpio)}}).rename_axis("Control").reset_index()
print("Tabla 1. Control de la tabla de partida (debe dar 1.212 días y 276 columnas, como en el notebook de preprocesamiento)")
display(tabla_simple(control, precision=0))

Tabla 1. Control de la tabla de partida (debe dar 1.212 días y 276 columnas, como en el notebook de preprocesamiento)


,Control,Valor
0,Días (filas) de la tabla limpia,1212
1,Columnas de la tabla limpia,276
2,Días de entrenamiento,848
3,Días de validación,182
4,Días de prueba,182


## 2. Qué pasa con las fechas de la tabla

Antes de calcular rezagos y variables de calendario hay que saber qué tan confiable es la columna `DIA`. En el EDA no se corrigieron las fechas: el ADR 0002 decidió tratar cada bloque de lecturas como un día distinto aunque repita la fecha, definir el día siguiente como la fila siguiente y conservar la fecha rotulada solo como dato informativo, con una consulta pendiente a la cátedra. Esta sección mide con la tabla cuánto se desvía esa fecha de un calendario normal. La primera tabla cuenta las fechas repetidas y las que faltan; la segunda muestra cuántos días hay entre la fecha de una fila y la de la siguiente (0 significa que repite la fecha, 1 que es el día siguiente y 2 o más que se saltea alguna); la tercera dice en qué días y meses ocurren las repeticiones y las faltas; la cuarta compara las filas de cada año con los días de su calendario; la quinta mide el desfase entre la fecha rotulada y la fecha que tendría la fila si todas fueran días consecutivos desde la primera (positivo significa que la etiqueta va adelantada), mes por mes y año por año; y la sexta mira de cerca cómo quedan rotulados los días de fin de año.

In [2]:
fechas = tabla_limpia["DIA"].reset_index(drop=True)
calendario = pd.date_range(fechas.min(), fechas.max())
cantidad_por_fecha = fechas.value_counts()
repetidas = cantidad_por_fecha[cantidad_por_fecha > 1].index.sort_values()
faltantes = calendario.difference(fechas)

resumen_fechas = pd.DataFrame({"Valor": {
    "Filas (un día cada una)": len(fechas),
    "Fechas distintas": fechas.nunique(),
    "Fechas que aparecen más de una vez": len(repetidas),
    "Filas de más por esas repeticiones": len(fechas) - fechas.nunique(),
    "Días del calendario entre la primera y la última fecha": len(calendario),
    "Días del calendario sin ninguna fila": len(faltantes)}}).rename_axis("Medida").reset_index()
print("Tabla 1. Fechas de la tabla completa")
display(tabla_simple(resumen_fechas, precision=0))

saltos = fechas.diff().dropna().dt.days.astype(int)
tabla_saltos = (saltos.value_counts().sort_index().rename("Filas")
                .rename_axis("Días entre una fecha y la de la fila siguiente").reset_index())
print("Tabla 2. Cómo cambia la fecha de una fila a la siguiente")
display(tabla_simple(tabla_saltos, precision=0))


def por_dia_y_mes(fechas_a_contar):
    # Cuenta en cuántas ocasiones ocurre cada día y mes, ordenado por mes y día
    cuenta = pd.Series(fechas_a_contar).dt.strftime("%d/%m").value_counts()
    cuenta = cuenta.rename("Veces").rename_axis("Día y mes").reset_index()
    return cuenta.sort_values("Día y mes", key=lambda s: s.str[3:] + s.str[:2]).reset_index(drop=True)


print("Tabla 3. Qué fechas se repiten y cuáles faltan, según su día y mes (Veces = cantidad de años en que ocurre)")
mostrar_lado_a_lado_simple([("Fechas que se repiten", por_dia_y_mes(repetidas)),
                            ("Fechas del calendario que faltan", por_dia_y_mes(faltantes))], precision=0)

por_anio = pd.DataFrame({"Filas": fechas.dt.year.value_counts(),
                         "Días del calendario": pd.Series(calendario.year).value_counts()}).sort_index()
por_anio["Diferencia"] = por_anio["Filas"] - por_anio["Días del calendario"]
print("Tabla 4. Filas y días del calendario por año de la fecha rotulada (2020 y 2023 son parciales)")
display(tabla_simple(por_anio.rename_axis("Año").reset_index(), precision=0))

posicion = pd.Series(pd.date_range(fechas.iloc[0], periods=len(fechas)))     # fecha si cada fila fuera el día siguiente
desfase = (fechas - posicion).dt.days
rango = pd.DataFrame({"anio": fechas.dt.year, "mes": fechas.dt.month, "desfase": desfase}).groupby(["mes", "anio"])["desfase"].agg(["min", "max"])
texto = rango.apply(lambda f: str(f["min"]) if f["min"] == f["max"] else f"{f['min']} a {f['max']}", axis=1)
tabla_desfase = texto.unstack("anio").fillna("sin datos").rename_axis("Mes").reset_index()
print("Tabla 5. Desfase en días entre la fecha rotulada y la fecha que tendría la fila si todas fueran consecutivas")
display(tabla_simple(tabla_desfase, precision=0))
print(f"Desfase mínimo y máximo de toda la tabla: {desfase.min()} y {desfase.max()} días.")
print(f"El patrón mes por mes de 2021 y el de 2022 es idéntico: {tabla_desfase[2021].equals(tabla_desfase[2022])}.")

dias_de_fin_de_anio = pd.date_range("2021-12-27", "2022-01-03")
filas_por_fecha = pd.DataFrame({
    f"Fin de {anio}": [int(((fechas.dt.year == anio + (f.month == 1)) & (fechas.dt.month == f.month) & (fechas.dt.day == f.day)).sum())
                       for f in dias_de_fin_de_anio] for anio in (2020, 2021, 2022)},
    index=dias_de_fin_de_anio.strftime("%d/%m")).rename_axis("Fecha rotulada").reset_index()
print("Tabla 6. Filas que tienen cada fecha de fin de año (0 significa que esa fecha no existe en la tabla)")
display(tabla_simple(filas_por_fecha, precision=0))

Tabla 1. Fechas de la tabla completa


,Medida,Valor
0,Filas (un día cada una),1212
1,Fechas distintas,1190
2,Fechas que aparecen más de una vez,16
3,Filas de más por esas repeticiones,22
4,Días del calendario entre la primera y la última fecha,1213
5,Días del calendario sin ninguna fila,23


Tabla 2. Cómo cambia la fecha de una fila a la siguiente


,Días entre una fecha y la de la fila siguiente,Filas
0,0,22
1,1,1173
2,2,12
3,3,1
4,4,3


Tabla 3. Qué fechas se repiten y cuáles faltan, según su día y mes (Veces = cantidad de años en que ocurre)


,Día y mes,Veces
0,28/02,3
1,30/04,3
2,30/06,3
3,30/09,4
4,30/11,3
,Día y mes,Veces
0,31/03,3
1,31/05,3
2,29/08,3
3,30/08,4


... (2 outputs left out)

Tabla 5. Desfase en días entre la fecha rotulada y la fecha que tendría la fila si todas fueran consecutivas


anio,Mes,2020,2021,2022,2023
0,1,sin datos,3,3,3
1,2,sin datos,0 a 3,0 a 3,0 a 3
2,3,sin datos,0,0,0
3,4,sin datos,0 a 1,0 a 1,0 a 1
4,5,sin datos,0,0,0
5,6,sin datos,0 a 1,0 a 1,0 a 1
6,7,0,0,0,0
7,8,0,0,0,0
8,9,2 a 3,2 a 3,2 a 3,1 a 2
9,10,2,2,2,1


Desfase mínimo y máximo de toda la tabla: 0 y 3 días.
El patrón mes por mes de 2021 y el de 2022 es idéntico: True.
Tabla 6. Filas que tienen cada fecha de fin de año (0 significa que esa fecha no existe en la tabla)


,Fecha rotulada,Fin de 2020,Fin de 2021,Fin de 2022
0,27/12,1,1,1
1,28/12,1,1,1
2,29/12,1,1,1
3,30/12,1,1,1
4,31/12,0,0,0
5,01/01,1,1,1
6,02/01,1,1,1
7,03/01,1,1,1


Las tablas muestran que las fechas no se corrigieron y que no se pueden tomar como un calendario exacto, pero tampoco son caóticas. Las repeticiones y las faltas ocurren siempre en los mismos días de fin de mes, un año completo tiene tantas filas como días (tabla 4), y el desfase respecto de la posición de la fila sigue un ciclo que en 2021 y en 2022 es idéntico mes por mes, con una diferencia de un día en los últimos meses de 2023 (tabla 5). El 31 de diciembre no existe en ningún año y la fecha pasa del 30/12 al 01/01, que es donde el desfase sube de 2 a 3 días (tabla 6). De ahí salen tres consecuencias para este notebook. Los rezagos y los promedios móviles se calculan por fila y no por fecha, porque las filas sí son días consecutivos. Las variables de estacionalidad (seno y coseno del día del año, fin de año) podrían heredar un error de pocos días. Como en 2021 y 2022 el ciclo se repite igual, la etiqueta funcionaría como una coordenada coherente entre años, pero eso no está comprobado para el año de prueba, donde el desfase difiere en un día en septiembre y octubre. Y el día de la semana es la variable más expuesta, porque un desfase de dos o tres días cambia por completo qué día de la semana es, así que su efecto se mide en la sección 6 con las dos fechas posibles.

## 3. Rezagos del frío: cuántos días hacia atrás

El objetivo de cada fila es el frío del día siguiente, así que el frío de hoy ya está en la tabla y la pregunta es cuántos días anteriores aportan algo. La tabla siguiente mide, con las filas de entrenamiento y la correlación de Spearman, cuánto se parecen el frío de mañana y el frío de k días antes de hoy, para rezagos de 0 (hoy), 1, 2 y 3 días y también para los de 7, 30 y 365 días que la consigna sugiere considerar. Se usa Spearman porque mide si dos series suben y bajan juntas sin exigir una relación lineal (ADR 0007). Cada rezago pierde sus primeras k filas, por eso la columna de días usados baja, y el de 365 días deja fuera casi la mitad del entrenamiento.

In [3]:
frio_entrenamiento = train_limpio[COLUMNA_FRIO]
objetivo_entrenamiento = train_limpio["objetivo"]
filas = []
for k in (0, 1, 2, 3, 7, 30, 365):
    pares = pd.DataFrame({"objetivo": objetivo_entrenamiento, "rezago": frio_entrenamiento.shift(k)}).dropna()
    filas.append({"Rezago (días)": k, "Días usados": len(pares),
                  "Spearman con el frío de mañana": pares["objetivo"].corr(pares["rezago"], method="spearman")})
print("Tabla 1. Parecido entre el frío de mañana y el frío de k días antes de hoy (solo entrenamiento)")
display(tabla_simple(pd.DataFrame(filas), precision=3))

Tabla 1. Parecido entre el frío de mañana y el frío de k días antes de hoy (solo entrenamiento)


,Rezago (días),Días usados,Spearman con el frío de mañana
0,0,817,0.847
1,1,813,0.752
2,2,812,0.731
3,3,811,0.728
4,7,808,0.732
5,30,786,0.586
6,365,462,0.646


La correlación cae del frío de hoy al de ayer y después los rezagos de uno, dos y tres días quedan con valores parecidos entre sí, así que traen información semejante pero no idéntica, y la selección posterior decidirá con cuáles se queda el modelo. El rezago de 7 días no cae respecto del de 3, pero eso no lo vuelve una buena variable: se parece al frío de la semana en general más que a un efecto semanal, y esa información la resume mejor el promedio de los últimos 7 días, que además tolera días faltantes. El rezago de 30 días se descarta porque es una fracción del ciclo estacional que ya capta el calendario. El de 365 días conserva una correlación apreciable, justamente porque repite la estación, pero deja sin dato a una parte grande del entrenamiento (la columna de días usados lo muestra) y exigiría un año de historia previa a cada día que se predice en `predict.py`, mientras que el seno y el coseno capturan la misma estacionalidad sin perder filas.

## 4. Estacionalidad: el mes y el día del año

La consigna pide codificar de forma cíclica las variables de calendario, y la estacionalidad es la señal más fuerte del problema. La celda siguiente usa `medir_efecto_por_grupo` sobre el frío de entrenamiento agrupado por mes: la primera tabla da el promedio y la mediana de cada mes y la segunda resume cuánto cambia el frío entre meses, con la diferencia entre el más alto y el más bajo, el porcentaje de la variación total que explica el mes y el p-valor de la prueba de Kruskal-Wallis, que se debe leer junto con el porcentaje porque los días consecutivos se parecen y el p-valor sale más chico de lo que correspondería.

In [4]:
entrenamiento = train_limpio[["DIA", COLUMNA_FRIO]].copy()
entrenamiento["mes"] = entrenamiento["DIA"].dt.month
resumen_mes, medidas_mes = medir_efecto_por_grupo(entrenamiento, "mes", COLUMNA_FRIO, orden=list(range(1, 13)))
print("Tabla 1. Frío por mes y medidas del efecto del mes (solo entrenamiento, días con dato)")
mostrar_lado_a_lado_simple([("Frío por mes (kW)", resumen_mes.reset_index().rename(columns={"mes": "Mes"}), {"precision": 0}),
                            ("Cuánto cambia el frío entre meses", medidas_mes.rename_axis("Medida").reset_index(), {"precision": 3})])

ejemplo = pd.DataFrame({"DIA": pd.to_datetime(["2021-06-30", "2021-12-14", "2021-12-23", "2021-12-30", "2021-12-31", "2022-01-05", "2022-01-06"]),
                        COLUMNA_FRIO: 1.0})
resultado = crear_variables_temporales(ejemplo)
resultado["Día que se predice"] = (resultado["DIA"] + pd.Timedelta(days=1)).dt.strftime("%d/%m/%Y")
print("Tabla 2. Ejemplo: seno, coseno y fin de año del día que se predice (el de mañana), para fechas elegidas")
display(tabla_simple(resultado[["Día que se predice", "dia_anio_seno", "dia_anio_coseno", "es_fin_de_anio"]], precision=3))

Tabla 1. Frío por mes y medidas del efecto del mes (solo entrenamiento, días con dato)


,Mes,Días,Promedio,Mediana
0,1,61,12822,12810
1,2,61,20010,19892
2,3,55,24345,23639
3,4,61,28375,27825
4,5,60,30235,30754
5,6,62,33174,33586
6,7,87,31624,31644
7,8,81,29324,29727
8,9,93,27345,26716
9,10,88,23256,23081


Tabla 2. Ejemplo: seno, coseno y fin de año del día que se predice (el de mañana), para fechas elegidas


,Día que se predice,dia_anio_seno,dia_anio_coseno,es_fin_de_anio
0,01/07/2021,0.011,-1.000,0
1,15/12/2021,-0.276,0.961,0
2,24/12/2021,-0.124,0.992,1
3,31/12/2021,-0.004,1.000,1
4,01/01/2022,0.017,1.000,1
5,06/01/2022,0.103,0.995,1
6,07/01/2022,0.120,0.993,0


El mes explica una parte grande de la variación del frío, así que la estacionalidad merece sus propias variables. Con el número del mes tal cual, diciembre (12) y enero (1) quedarían lejos aunque son meses vecinos y de consumo parecido; con el seno y el coseno del día del año quedan cerca, como se ve en la segunda tabla, donde el 31 de diciembre y el 1 de enero tienen valores casi iguales. Se usa el día del año y no el mes porque el mes da el mismo valor a todos los días y salta el día 1, mientras que el día del año cambia de forma gradual. Se calcula sobre el día que se predice (mañana), que ya se conoce hoy y es la fecha del objetivo, como la fecha de hoy más un día: así lo puede obtener `predict.py` aunque la fila de mañana todavía no exista.

## 5. Fin de año

El EDA mostró que el consumo cae por debajo de lo que explica la estación alrededor de las fiestas, probablemente porque la planta se detiene. Esta sección mide esa caída. La primera tabla da, para cada fecha rotulada entre el 15 de diciembre y el 15 de enero, la mediana del frío de entrenamiento y en cuántos años hay dato, e indica si esa fecha cae dentro de la ventana elegida, que va del 24 de diciembre al 6 de enero ambos incluidos. La segunda compara la mediana dentro de la ventana con la del resto de diciembre y enero y con la del resto del año.

In [5]:
entrenamiento["dia_mes"] = entrenamiento["DIA"].dt.strftime("%m-%d")
con_dato = entrenamiento.dropna(subset=[COLUMNA_FRIO])

referencia = pd.Series(pd.date_range("2021-12-15", "2022-01-15"))        # año cualquiera sin 29 de febrero, solo para ordenar
por_fecha = con_dato.groupby("dia_mes")[COLUMNA_FRIO].agg(["median", "count"]).reindex(referencia.dt.strftime("%m-%d"))
tabla_fecha = pd.DataFrame({"Fecha rotulada": referencia.dt.strftime("%d/%m").to_numpy(),
                            "Mediana del frío (kW)": por_fecha["median"].to_numpy(),
                            "Años con dato": por_fecha["count"].to_numpy(),
                            "¿Dentro de la ventana?": np.where(es_fin_de_anio(referencia), "sí", "no")})
print(f"Tabla 1. Frío por fecha rotulada, del 15/12 al 15/01 (solo entrenamiento). Ventana: del {INICIO_FIN_DE_ANIO[1]}/{INICIO_FIN_DE_ANIO[0]} al {FIN_FIN_DE_ANIO[1]}/{FIN_FIN_DE_ANIO[0]}")
display(tabla_simple(tabla_fecha, precision=0))

en_ventana = es_fin_de_anio(con_dato["DIA"])
diciembre_enero = con_dato["mes"].isin([12, 1])
grupos = {"Dentro de la ventana de fin de año": con_dato[en_ventana],
          "Resto de diciembre y enero": con_dato[~en_ventana & diciembre_enero],
          "Resto del año": con_dato[~diciembre_enero]}
comparacion = pd.DataFrame({"Días": {k: len(v) for k, v in grupos.items()},
                            "Mediana del frío (kW)": {k: v[COLUMNA_FRIO].median() for k, v in grupos.items()}}).rename_axis("Grupo de días").reset_index()
print("Tabla 2. Frío dentro y fuera de la ventana de fin de año (solo entrenamiento)")
display(tabla_simple(comparacion, precision=0))

Tabla 1. Frío por fecha rotulada, del 15/12 al 15/01 (solo entrenamiento). Ventana: del 24/12 al 6/1


,Fecha rotulada,Mediana del frío (kW),Años con dato,¿Dentro de la ventana?
0,15/12,15393,2,no
1,16/12,13943,2,no
2,17/12,11742,2,no
3,18/12,12387,2,no
4,19/12,10114,2,no
5,20/12,9334,2,no
6,21/12,11506,2,no
7,22/12,10492,2,no
8,23/12,12445,2,no
9,24/12,13964,2,sí


Tabla 2. Frío dentro y fuera de la ventana de fin de año (solo entrenamiento)


,Grupo de días,Días,Mediana del frío (kW)
0,Dentro de la ventana de fin de año,25,9751
1,Resto de diciembre y enero,96,13089
2,Resto del año,710,27019


La ventana captura la zona de consumo más bajo y la recuperación de los primeros días de enero, y la mediana dentro de ella queda claramente por debajo de la del resto de diciembre y enero, que ya es de por sí el período de menor consumo del año. Eso justifica una variable propia además del seno y el coseno, que describen una curva suave y no pueden representar una caída tan concentrada. Hay cuatro cautelas. Solo hay dos fines de año completos en entrenamiento (la columna de años con dato lo muestra), así que el borde exacto de la ventana tiene poca evidencia y se prefirió una ventana simple y fácil de explicar en lugar de una afinada día por día. Unos días antes, entre el 19 y el 22 de diciembre, hay otra caída que la ventana deja afuera, y esa parte la sigue captando el seno y el coseno. Las fechas son las rotuladas y podrían tener el desfase de la sección 2, que en 2021 y 2022 se repite igual cada año. Y la variable indica si el día que se predice (mañana) cae en la ventana, que es lo que se conoce hoy.

## 6. Tendencia entre años y día de la semana

Dos ideas se miden antes de decidir: si conviene una variable de año, porque a simple vista los picos parecen bajar de un año a otro, y si el día de la semana aporta algo. Para el año, la primera tabla muestra cuántos días con dato y cuántos meses distintos tiene cada año en entrenamiento y su mediana, y la segunda la mediana de cada mes en cada año, para comparar el mismo mes entre años y no meses distintos. La tercera mide cuánto de la variación explica el año tal cual y cuánto explica una vez descontada la mediana de su mes, que es la comparación justa porque cada año incluye meses distintos.

In [6]:
entrenamiento["anio"] = entrenamiento["DIA"].dt.year
con_dato = entrenamiento.dropna(subset=[COLUMNA_FRIO])

por_anio = con_dato.groupby("anio").agg(**{"Días con dato": (COLUMNA_FRIO, "count"), "Meses distintos": ("mes", "nunique"),
                                           "Mediana del frío (kW)": (COLUMNA_FRIO, "median")}).rename_axis("Año").reset_index()
mes_por_anio = con_dato.pivot_table(index="mes", columns="anio", values=COLUMNA_FRIO, aggfunc="median").rename_axis("Mes").reset_index()
mes_por_anio.columns = [str(c) for c in mes_por_anio.columns]
print("Tabla 1 y 2. Frío por año y mediana de cada mes en cada año (solo entrenamiento)")
mostrar_lado_a_lado_simple([("Por año", por_anio, {"precision": 0}), ("Mediana por mes y año (kW)", mes_por_anio, {"precision": 0})])

entrenamiento["desvio_del_mes"] = entrenamiento[COLUMNA_FRIO] - entrenamiento.groupby("mes")[COLUMNA_FRIO].transform("median")
resumen_a, medidas_a = medir_efecto_por_grupo(entrenamiento, "anio", COLUMNA_FRIO)
resumen_d, medidas_d = medir_efecto_por_grupo(entrenamiento, "anio", "desvio_del_mes")
medidas_anio = pd.DataFrame({"El año tal cual": medidas_a["Valor"], "Descontando la mediana de su mes": medidas_d["Valor"]}).rename_axis("Medida").reset_index()
medidas_anio["Descontando la mediana de su mes"] = medidas_anio["Descontando la mediana de su mes"].astype(object)
medidas_anio.iloc[1, 2] = "no aplica"          # el desvío promedia casi 0, un % sobre su promedio no tiene sentido
print("Tabla 3. Cuánto explica el año del frío (solo entrenamiento)")
display(tabla_simple(medidas_anio, precision=3))
anios_completos = por_anio.loc[por_anio["Meses distintos"] == 12, "Año"].tolist()
print(f"Años de entrenamiento con los 12 meses: {anios_completos}. Año de la prueba: {test_limpio['DIA'].dt.year.unique().tolist()}.")

Tabla 1 y 2. Frío por año y mediana de cada mes en cada año (solo entrenamiento)

,Año,Días con dato,Meses distintos,Mediana del frío (kW)
0,2020,180,6,25769
1,2021,357,12,26489
2,2022,294,10,24789
,Mes,2020,2021,2022
0,1,nan,12798,12965
1,2,nan,19898,19890
2,3,nan,24736,22408
3,4,nan,28086,27825
4,5,nan,30793,30320
5,6,nan,34945,32442


Tabla 3. Cuánto explica el año del frío (solo entrenamiento)


,Medida,El año tal cual,Descontando la mediana de su mes
0,Diferencia entre el grupo más alto y el más bajo,328.215,2108.646
1,"Esa diferencia, como % del promedio general",1.322,no aplica
2,% de la variación total explicado por los grupos,0.031,5.126
3,p-valor de la prueba de Kruskal-Wallis,0.534,0.000


Años de entrenamiento con los 12 meses: [2021]. Año de la prueba: [2023].


Estas tablas dicen algo distinto de lo que sugiere la impresión visual de picos cada vez más bajos. El año tal cual explica una fracción despreciable de la variación, porque 2020 solo tiene el segundo semestre y 2022 solo llega a octubre, de modo que cada año promedia meses distintos y la estación domina la comparación. Al comparar el mismo mes entre años (segunda tabla) no hay una tendencia en un solo sentido: entre 2020 y 2021 algunos meses suben y otros bajan, y en 2022 casi todos los meses quedan por debajo de los de 2021, con diferencias grandes hacia el final. Descontada la estación, el año explica una fracción pequeña pero no nula (tercera tabla), y se concentra en ese cambio de nivel de 2022 más que en una pendiente. Una variable de año no sirve para representarlo: el entrenamiento tiene un solo año completo y el año de la prueba es uno que el modelo nunca vio, un modelo de árboles no puede extrapolar a un valor de año nuevo y uno lineal extrapolaría una tendencia que con tan pocos años no se distingue de una coincidencia. Por eso no se incluye. Si el nivel del consumo se desplaza, lo siguen los rezagos y el promedio móvil de los últimos 7 días sin fijar ningún año.

Para el día de la semana, como las fechas pueden estar desfasadas (sección 2), se mide el efecto de dos maneras: con el día de la semana de la fecha rotulada y con el de la fecha que tendría la fila si todas fueran días consecutivos desde la primera. Si la conclusión es la misma con las dos, no depende de cuál sea la correcta. La primera tabla da la mediana del frío por día de la semana con cada fecha y la segunda las medidas del efecto.

In [7]:
nombres_dias = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]
fecha_por_posicion = pd.Series(pd.date_range(tabla_limpia["DIA"].iloc[0], periods=len(tabla_limpia)), index=tabla_limpia.index)
entrenamiento["dia_semana_rotulado"] = entrenamiento["DIA"].dt.dayofweek
entrenamiento["dia_semana_posicion"] = fecha_por_posicion.loc[entrenamiento.index].dt.dayofweek

resumen_rot, medidas_rot = medir_efecto_por_grupo(entrenamiento, "dia_semana_rotulado", COLUMNA_FRIO, orden=list(range(7)))
resumen_pos, medidas_pos = medir_efecto_por_grupo(entrenamiento, "dia_semana_posicion", COLUMNA_FRIO, orden=list(range(7)))
medianas = pd.DataFrame({"Fecha rotulada": resumen_rot["Mediana"].to_numpy(), "Fecha según la posición": resumen_pos["Mediana"].to_numpy()},
                        index=nombres_dias).rename_axis("Día de la semana").reset_index()
medidas_semana = pd.DataFrame({"Fecha rotulada": medidas_rot["Valor"], "Fecha según la posición": medidas_pos["Valor"]}).rename_axis("Medida").reset_index()
print("Tabla 1 y 2. Mediana del frío por día de la semana y medidas del efecto, con las dos fechas (solo entrenamiento)")
mostrar_lado_a_lado_simple([("Mediana del frío por día (kW)", medianas, {"precision": 0}),
                            ("Cuánto cambia el frío entre días de la semana", medidas_semana, {"precision": 3})])

Tabla 1 y 2. Mediana del frío por día de la semana y medidas del efecto, con las dos fechas (solo entrenamiento)


,Día de la semana,Fecha rotulada,Fecha según la posición
0,lunes,25209,26382
1,martes,26416,26042
2,miércoles,26935,26420
3,jueves,26585,25354
4,viernes,24821,23056
5,sábado,24569,25639
6,domingo,25879,27019
,Medida,Fecha rotulada,Fecha según la posición
0,Diferencia entre el grupo más alto y el más bajo,2064.925,2882.032
1,"Esa diferencia, como % del promedio general",8.320,11.612


Con cualquiera de las dos fechas el día de la semana explica una fracción muy chica de la variación, mucho menor que la del mes, así que no se espera que el indicador de fin de semana aporte. Con una de las fechas la prueba de Kruskal-Wallis da un p-valor bajo, pero hay que leerlo junto con el porcentaje explicado y recordar que los días consecutivos se parecen, lo que lo achica de más. Se construye el indicador de todos modos, solo para probarlo: la consigna pide que, si el efecto del día de la semana resulta despreciable, se diga con el número y se descarte, y la selección de la etapa siguiente lo confirmará o lo descartará con esa evidencia.

## 7. Construcción de las variables y controles

Con las decisiones tomadas se aplica `crear_variables_temporales` a la tabla limpia completa, en orden cronológico, y se vuelve a dividir. Sale una tabla con las columnas de partida más diez nuevas: tres rezagos del frío (ayer, hace 2 y hace 3 días), el promedio y el desvío estándar móviles de los últimos siete días con tolerancia de hasta cuatro días faltantes (se pide un mínimo de tres con dato), la diferencia entre hoy y ayer, el seno y el coseno del día del año del día que se predice, y los indicadores de fin de año y de fin de semana de ese mismo día. Los valores quedan en kW, sin transformar ni escalar, porque eso se hace dentro del `Pipeline` (ADR 0010). La primera tabla describe cada variable y cuántos vacíos tiene en cada tramo; los vacíos vienen de los días inválidos del frío y de los primeros días de la tabla, y se tratan más adelante dentro del `Pipeline`, nunca acá.

In [8]:
tabla_con_temporales = crear_variables_temporales(tabla_limpia)
train_t, val_t, test_t = dividir_por_fecha(tabla_con_temporales)

descripcion = {
    "frio_ayer": "Frío de ayer (la fila anterior)",
    "frio_hace_2_dias": "Frío de hace 2 días",
    "frio_hace_3_dias": "Frío de hace 3 días",
    "frio_media_7_dias": f"Promedio del frío de hoy y los {VENTANA_DIAS - 1} días anteriores (mínimo {MIN_DIAS_CON_DATO} con dato)",
    "frio_desvio_7_dias": f"Desvío estándar del frío en esos {VENTANA_DIAS} días (mínimo {MIN_DIAS_CON_DATO} con dato)",
    "frio_hoy_menos_ayer": "Frío de hoy menos frío de ayer",
    "dia_anio_seno": "Seno del día del año del día que se predice",
    "dia_anio_coseno": "Coseno del día del año del día que se predice",
    "es_fin_de_anio": "1 si el día que se predice cae del 24/12 al 6/01",
    "es_fin_de_semana": "1 si el día que se predice es sábado o domingo"}
resumen_nuevas = pd.DataFrame({
    "Qué es": pd.Series(descripcion),
    "Mínimo": train_t[NOMBRES_NUEVAS].min(), "Máximo": train_t[NOMBRES_NUEVAS].max(),
    "Vacíos en entrenamiento": train_t[NOMBRES_NUEVAS].isna().sum(),
    "Vacíos en validación": val_t[NOMBRES_NUEVAS].isna().sum(),
    "Vacíos en prueba": test_t[NOMBRES_NUEVAS].isna().sum()}).rename_axis("Variable").reset_index()
print("Tabla 1. Variables temporales creadas, con su rango y sus vacíos por tramo")
display(tabla_simple(resumen_nuevas, precision=2))

columnas_de_muestra = ["DIA", COLUMNA_FRIO, "frio_ayer", "frio_hace_2_dias", "frio_hace_3_dias", "frio_media_7_dias", "frio_desvio_7_dias", "frio_hoy_menos_ayer"]
muestra = train_t[columnas_de_muestra].iloc[:9].copy()
muestra["DIA"] = muestra["DIA"].dt.strftime("%d/%m/%Y")
print("Tabla 2. Primeras 9 filas de entrenamiento: se ve cómo los rezagos desplazan el frío y cómo la ventana móvil empieza a dar valor desde el tercer día con dato")
display(tabla_simple(muestra, precision=0))

Tabla 1. Variables temporales creadas, con su rango y sus vacíos por tramo

,Variable,Qué es,Mínimo,Máximo,Vacíos en entrenamiento,Vacíos en validación,Vacíos en prueba
0,frio_ayer,Frío de ayer (la fila anterior),2310.00,48046.00,18,5,2
1,frio_hace_2_dias,Frío de hace 2 días,2310.00,48046.00,19,5,2
2,frio_hace_3_dias,Frío de hace 3 días,2310.00,48046.00,20,5,2
3,frio_media_7_dias,Promedio del frío de hoy y los 6 días anteriores (mínimo 3 con dato),3909.83,37575.00,2,0,0
4,frio_desvio_7_dias,Desvío estándar del frío en esos 7 días (mínimo 3 con dato),542.91,9741.78,2,0,0
5,frio_hoy_menos_ayer,Frío de hoy menos frío de ayer,-15992.00,17881.00,32,10,5
6,dia_anio_seno,Seno del día del año del día que se predice,-1.00,1.00,0,0,0
7,dia_anio_coseno,Coseno del día del año del día que se predice,-1.00,1.00,0,0,0
8,es_fin_de_anio,1 si el día que se predice cae del 24/12 al 6/01,0.00,1.00,0,0,0
9,es_fin_de_semana,1 si el día que se predice es sábado o domingo,0.00,1.00,0,0,0


Tabla 2. Primeras 9 filas de entrenamiento: se ve cómo los rezagos desplazan el frío y cómo la ventana móvil empieza a dar valor desde el tercer día con dato


,DIA,Frio (Kw),frio_ayer,frio_hace_2_dias,frio_hace_3_dias,frio_media_7_dias,frio_desvio_7_dias,frio_hoy_menos_ayer
0,01/07/2020,14325,nan,nan,nan,nan,nan,nan
1,02/07/2020,23954,14325,nan,nan,nan,nan,9629
2,03/07/2020,28268,23954,14325,nan,22182,7138,4314
3,04/07/2020,24246,28268,23954,14325,22698,5919,-4022
4,05/07/2020,29885,24246,28268,23954,24136,6050,5639
5,06/07/2020,24449,29885,24246,28268,24188,5413,-5436
6,07/07/2020,24435,24449,29885,24246,24223,4942,-14
7,08/07/2020,33401,24435,24449,29885,26948,3670,8966
8,09/07/2020,30921,33401,24435,24449,27944,3668,-2480


Tres controles comprueban que la función se comporta como se dijo. El primero verifica que las columnas de la tabla de partida quedaron idénticas y que no cambió la cantidad de filas. El segundo prueba el caso más peligroso, la fuga de información: se reemplaza el frío por ceros desde la fila 600 en adelante y se verifica que las variables de las filas anteriores no cambian, es decir, que ninguna mira hacia el futuro. El tercero confirma que los rezagos son por fila y no por fecha, comparando cada rezago con el frío de la fila correspondiente.

In [9]:
columnas_originales_iguales = tabla_con_temporales[tabla_limpia.columns].equals(tabla_limpia)
mismas_filas = len(tabla_con_temporales) == len(tabla_limpia) and tabla_con_temporales.index.equals(tabla_limpia.index)

alterada = tabla_limpia.copy()
alterada.iloc[600:, alterada.columns.get_loc(COLUMNA_FRIO)] = 0.0              # se destruye el futuro desde la fila 600
con_alteracion = crear_variables_temporales(alterada)
variables_del_frio = [c for c in NOMBRES_NUEVAS if c.startswith("frio_")]
sin_fuga = con_alteracion[variables_del_frio].iloc[:600].equals(tabla_con_temporales[variables_del_frio].iloc[:600])

frio = tabla_limpia[COLUMNA_FRIO].reset_index(drop=True)
rezagos_por_fila = all(tabla_con_temporales[f"frio_{n}"].reset_index(drop=True).equals(frio.shift(k))
                       for n, k in (("ayer", 1), ("hace_2_dias", 2), ("hace_3_dias", 3)))

controles = pd.DataFrame({"Resultado": {
    "Las columnas de partida no cambiaron": columnas_originales_iguales,
    "Misma cantidad de filas y mismo índice": mismas_filas,
    "Cambiar el frío desde la fila 600 no altera las variables de las filas anteriores": sin_fuga,
    "Los rezagos son el frío de la fila anterior, de dos y de tres filas atrás": rezagos_por_fila}}).rename_axis("Control").reset_index()
print("Tabla 1. Controles de la función (los cuatro deben dar True)")
display(tabla_simple(controles))
print(f"Forma de la tabla con variables temporales: {tabla_con_temporales.shape}. Entrenamiento {train_t.shape}, validación {val_t.shape}, prueba {test_t.shape}.")

Tabla 1. Controles de la función (los cuatro deben dar True)


,Control,Resultado
0,Las columnas de partida no cambiaron,True
1,Misma cantidad de filas y mismo índice,True
2,Cambiar el frío desde la fila 600 no altera las variables de las filas anteriores,True
3,"Los rezagos son el frío de la fila anterior, de dos y de tres filas atrás",True


Forma de la tabla con variables temporales: (1212, 286). Entrenamiento (848, 286), validación (182, 286), prueba (182, 286).


## 8. Primera mirada a las variables nuevas

Para cerrar se mide, solo con entrenamiento, cuánto se parece cada variable nueva al frío de mañana (primera tabla) y cuánto se parecen entre sí las variables del frío (segunda tabla). No son una selección: la etapa de selección usa dos métodos distintos con `TimeSeriesSplit`. Sirven para anticipar qué variables traen información parecida y, por lo tanto, cuáles probablemente se descarten entre sí.

In [10]:
base = train_t.dropna(subset=["objetivo"])
correlacion_objetivo = base[NOMBRES_NUEVAS].corrwith(base["objetivo"], method="spearman")
tabla_objetivo = (correlacion_objetivo.rename("Spearman con el frío de mañana").rename_axis("Variable").reset_index()
                  .sort_values("Spearman con el frío de mañana", key=abs, ascending=False))
print("Tabla 1. Parecido de cada variable nueva con el frío de mañana (solo entrenamiento, días con objetivo)")
display(tabla_simple(tabla_objetivo, precision=3))

entre_si = train_t[["frio_ayer", "frio_hace_2_dias", "frio_hace_3_dias", "frio_media_7_dias", "frio_desvio_7_dias", "frio_hoy_menos_ayer"]].corr(method="spearman")
print("Tabla 2. Parecido entre las variables del frío (Spearman, solo entrenamiento)")
display(tabla_simple(entre_si.rename_axis("Variable").reset_index(), precision=2))

Tabla 1. Parecido de cada variable nueva con el frío de mañana (solo entrenamiento, días con objetivo)


,Variable,Spearman con el frío de mañana
3,frio_media_7_dias,0.834
7,dia_anio_coseno,-0.801
0,frio_ayer,0.752
1,frio_hace_2_dias,0.731
2,frio_hace_3_dias,0.728
8,es_fin_de_anio,-0.273
5,frio_hoy_menos_ayer,0.177
4,frio_desvio_7_dias,0.118
6,dia_anio_seno,0.020
9,es_fin_de_semana,-0.020


Tabla 2. Parecido entre las variables del frío (Spearman, solo entrenamiento)


,Variable,frio_ayer,frio_hace_2_dias,frio_hace_3_dias,frio_media_7_dias,frio_desvio_7_dias,frio_hoy_menos_ayer
0,frio_ayer,1.00,0.85,0.75,0.90,0.10,-0.26
1,frio_hace_2_dias,0.85,1.00,0.85,0.90,0.11,-0.16
2,frio_hace_3_dias,0.75,0.85,1.00,0.90,0.12,-0.05
3,frio_media_7_dias,0.90,0.90,0.90,1.00,0.14,-0.03
4,frio_desvio_7_dias,0.10,0.11,0.12,0.14,1.00,0.01
5,frio_hoy_menos_ayer,-0.26,-0.16,-0.05,-0.03,0.01,1.00


Las variables temporales quedan creadas y controladas. Entre las que se descartan por ahora, sin construirlas, están los rezagos de 7, 30 y 365 días y la variable de año, con la evidencia de las secciones 3 y 6. El indicador de fin de semana se construye solo para probarlo y probablemente la selección lo descarte, lo que se dirá con el número cuando corresponda. El paso siguiente de la ingeniería de características son las variables de planta, y después la integración de todo en una función única, `construir_variables`, antes de la selección.